In [12]:
# https://colab.research.google.com/drive/1JMLa53HDuA-i7ZBmqV7ZnA3c_fvtXnx-?usp=sharing#scrollTo=wJpXpmjEYC_T
# https://www.bilibili.com/video/BV1BbFaeVE4W  PyTorch手搓Transformer
# https://github.com/hankinghu/literature-books/tree/master

In [1]:

import torch
import torch.nn as nn
from torch.nn import functional as F
import textwrap
import random

# 超参数
file_name="sanguo-all.txt"
batch_size = 64 # how many independent sequences will we process in parallel?
block_size = 128 # what is the maximum context length for predictions?
wrap_width = 50
max_iters = 15000
eval_interval = 1000
learning_rate = 1e-3
device = 'cuda' if torch.cuda.is_available() else 'cpu'
eval_iters = 200
n_embd = 128
n_head = 4
head_size = n_embd // n_head
n_layer = 4
dropout = 0.1
# ------------

torch.manual_seed(1337)


In [2]:
# wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
with open(file_name, 'r', encoding='utf-8') as f:
    text = f.read()

# here are all the unique characters that occur in this text
chars = sorted(list(set(text)))
vocab_size = len(chars)
# create a mapping from characters to integers
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }
encode = lambda s: [stoi[c] for c in s] # encoder: take a string, output a list of integers
decode = lambda l: ''.join([itos[i] for i in l]) # decoder: take a list of integers, output a string

# Train and test splits
data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9*len(data)) # first 90% will be train, rest val
train_data = data[:n]
val_data = data[n:]

# data loading
def get_batch(split):
    # generate a small batch of data of inputs x and targets y
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y


In [3]:
# Head类 注意力机制
class Head(nn.Module):
    def __init__(self, head_size):
        super().__init__ ()
        self.query = nn.Linear(n_embd,head_size,bias=False) # 线性变换层
        self.key = nn.Linear(n_embd,head_size,bias=False) # 线性变换层
        self.value = nn.Linear(n_embd,head_size,bias=False) # 线性变换层
        self.register_buffer("tril",torch.tril(torch.ones(block_size,block_size)))#不可训练的,结构(约等于常量)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B,T,C = x.shape
        q= self.query(x)  #(B,T, head size)
        k=self.key(x)
        wei =q @ k.transpose(-2,-1)*k.shape[-1]**-0.5 #注意力方阵(B，T，T)
        wei = wei.masked_fill(self.tril == 0,float("-inf"))# 掩码填充
        wei =F.softmax(wei,dim=-1)
        wei = self.dropout(wei)  # 随机去掉(归零)一些值，增加网络的稳定性
        v = self.value(x)
        out = wei @ v   #(B, T, head size)
        return out

In [4]:
class MultiHeadAttention(nn.Module):
    """ multiple heads of self-attention in parallel """

    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.dropout(self.proj(out))
        return out


In [5]:
class FeedFoward(nn.Module):
    """ a simple linear layer followed by a non-linearity """

    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)

class Block(nn.Module):
    """ Transformer block: communication followed by computation """

    def __init__(self, n_embd, n_head):
        # n_embd: embedding dimension, n_head: the number of heads we'd like
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size) # 多头注意力
        self.ffwd = FeedFoward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.sa(self.ln1(x)) # 残差多头注意力网络
        x = x + self.ffwd(self.ln2(x)) # 残差线性前反馈层
        return x


In [6]:
# 语言模型
class LanguageModel(nn.Module):
    def __init__ (self):
        super().__init__ ()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(*[Block(n_embd, n_head=n_head) for _ in range(n_layer)])
        # self.blocks = Block(n_embd, n_head=n_head)
        self.ln_f = nn.LayerNorm(n_embd) # final layer norm
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self,idx,targets=None):
        B,T=idx.shape     #(B,T)B= batch_size,T= block_size，数据为token(整数)形式
        token_embd=self.token_embedding_table(idx)
        position_idx= torch.arange(T,device=device)
        position_embd = self.position_embedding_table(position_idx)
        x=token_embd + position_embd #(B,T,n embd)
        x = self.blocks(x) # (B,T,C)
        x = self.ln_f(x) # (B,T,C)
        logits = self.lm_head(x) # (B,T,vocab_size)


        # head_out = self.head(x)  #添加注意力头
        # logits =self.network(head_out)  #(B，T，vocab size)
        if targets is None:
            loss = None
        else:
            B, T, C= logits.shape
            logits =logits.view(B*T, C)  #摊平
            targets = targets.view(B*T)
            loss=F.cross_entropy(logits, targets)

        # B,T=idx.shape #B= batch size,T= block size，数据为token(整数)形式
        # random_tensor = torch.rand(B,T,vocab_size,device=device) #
        # logits = random_tensor /random_tensor.sum(dim=-1, keepdim=True)
        # loss = None
        return logits, loss
    
    def generate(self, token_sequ, max_new_tokens):
        # token_sequ已知的上文,max_new_tokens是续写的长度(B，T)
        for _ in range(max_new_tokens):
            tokens_input = token_sequ[:, -block_size: ]
            logits, loss = self.forward(tokens_input)  # logits,(B, T, vocab size)
            logits = logits[:,-1,:] #只取字符串最后一个,(概率分布向量格式)
            probs =F.softmax(logits,dim=-1)
            token_next = torch.multinomial(probs,num_samples=1)# 概率分布向量-->one-hot 向量-->整数token
            token_sequ =torch.cat((token_sequ, token_next), dim=1)
        new_tokens =token_sequ[:,-max_new_tokens:]
        return new_tokens

In [7]:
#--损失评测--------
@torch.no_grad()   #不做梯度计算的decorator,作用域为整个函数
def estimate_loss(model):
    out = {}
    model.eval()  #把模型转化为evaluate模式(默认模式是train)
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)  # 建立一个初始值为0的容器,用于储存loss值
        for k in range(eval_iters):
            X, Y = get_batch(split)  # split是一个字符串,用来控制get_batch()函数的行为
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()  # out是含有两个元素的字典，一个是train，一个是val，每个元素对应一个loss的平均值
    model.train() # 再转化为训练模式(如果之前没有转为evaluate模式,则不需要这一步,因为模型建立后默认为训练模式)
    return out

In [8]:
def main():
    print(f"训练内容:{file_name}")
    model =LanguageModel()#实例化
    model = model.to(device)
    print(sum(p.numel()for p in model.parameters())/1e6,'M parameters')# 打印有多少个参数
    #设定一个优化器
    optimizer = torch.optim.Adam(model.parameters(),lr=learning_rate)
    # 训练循环
    for i in range(max_iters):
        if i % eval_interval ==0 or i==max_iters - 1:
            losses =estimate_loss(model)
            print(f"step {i}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")
        #取样
        xb,yb = get_batch("train")
        logits, loss=model(xb, yb)   #前馈运算
        optimizer.zero_grad()   #把旧的梯度归零
        loss.backward()   #反向传播,计算新的梯度
        optimizer.step()  #做一步优化计算

    print("训练结束，下面开始生成内容")
    max_new_tokens =200
    start_idx = random.randint(0, len(val_data)-block_size-max_new_tokens)
    #上文内容
    context = torch.zeros((1, block_size), dtype=torch.long, device=device)# (B, T)B = 1,T = block size
    context[0,:]=val_data[start_idx:start_idx+block_size]
    context_str =decode(context[0].tolist())#一阶张量
    wrapped_context_str = textwrap.fill(context_str, width=wrap_width)
    #真实下文
    real_next_tokens = torch.zeros((1,max_new_tokens), dtype=torch.long, device=device)
    real_next_tokens[0, :]= val_data[start_idx+block_size: start_idx+block_size+max_new_tokens]
    real_next_tokens_str = decode(real_next_tokens[0].tolist())# 一阶张量
    wrapped_real_next_tokens_str = textwrap.fill(real_next_tokens_str, width=wrap_width)
    #生成下文
    generated_tokens = model.generate(context, max_new_tokens)
    generated_str =decode(generated_tokens[0].tolist())
    wrapped_generated_str = textwrap.fill(generated_str, width=wrap_width)

    print("---------上文内容---------:")
    print(wrapped_context_str)
    print("---------真实上文内容---------:")
    print(wrapped_real_next_tokens_str)
    print("---------生成内容---------:")
    print(wrapped_generated_str)


main()

训练内容:sanguo-all.txt
1.82694 M parameters
step 0: train loss 8.4513, val loss 8.4476
step 1000: train loss 3.9267, val loss 4.7499
step 2000: train loss 3.4746, val loss 4.7091
step 3000: train loss 3.2125, val loss 4.7510
step 4000: train loss 3.0395, val loss 4.7872
step 5000: train loss 2.8987, val loss 4.8776
step 6000: train loss 2.7783, val loss 4.9568
step 7000: train loss 2.6851, val loss 5.0320
step 8000: train loss 2.6037, val loss 5.0706
step 9000: train loss 2.5332, val loss 5.1718
step 10000: train loss 2.4570, val loss 5.2480
step 11000: train loss 2.3889, val loss 5.3023
step 12000: train loss 2.3348, val loss 5.3693
step 13000: train loss 2.2783, val loss 5.4300
step 14000: train loss 2.2222, val loss 5.5206
step 14999: train loss 2.1790, val loss 5.5615
训练结束，下面开始生成内容
---------上文内容---------:
防不测。 却说姜维在钟提大设筵宴，会集诸将，商议伐魏之事。令史樊建谏曰：“将军屡出，未获全功；今日洮
西之捷，魏人已服威名，何故又欲出也？万一不利，前功尽弃。”维曰：“汝等只知魏国地宽人广，急不可得；却
不知攻魏者有五可胜。”众问之，维答曰：“彼洮西一败，挫尽
---------真实上文内容---------:
锐气，吾兵虽退，不曾损折：今若进兵，一可胜也。吾兵

In [7]:

class Head(nn.Module):
    """ one head of self-attention """

    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B,T,C = x.shape
        k = self.key(x)   # (B,T,C)
        q = self.query(x) # (B,T,C)
        # compute attention scores ("affinities")
        wei = q @ k.transpose(-2,-1) * C**-0.5 # (B, T, C) @ (B, C, T) -> (B, T, T)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf')) # (B, T, T)
        wei = F.softmax(wei, dim=-1) # (B, T, T)
        wei = self.dropout(wei)
        # perform the weighted aggregation of the values
        v = self.value(x) # (B,T,C)
        out = wei @ v # (B, T, T) @ (B, T, C) -> (B, T, C)
        return out

class MultiHeadAttention(nn.Module):
    """ multiple heads of self-attention in parallel """

    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.dropout(self.proj(out))
        return out

class FeedFoward(nn.Module):
    """ a simple linear layer followed by a non-linearity """

    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)

class Block(nn.Module):
    """ Transformer block: communication followed by computation """

    def __init__(self, n_embd, n_head):
        # n_embd: embedding dimension, n_head: the number of heads we'd like
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedFoward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x


In [ ]:

# super simple bigram model
class BigramLanguageModel(nn.Module):

    def __init__(self):
        super().__init__()
        # each token directly reads off the logits for the next token from a lookup table
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks = Block(n_embd, n_head=n_head)
        # self.blocks = nn.Sequential(*[Block(n_embd, n_head=n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd) # final layer norm
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape

        # idx and targets are both (B,T) tensor of integers
        tok_emb = self.token_embedding_table(idx) # (B,T,C)
        pos_emb = self.position_embedding_table(torch.arange(T, device=device)) # (T,C)
        x = tok_emb + pos_emb # (B,T,C)
        x = self.blocks(x) # (B,T,C)
        x = self.ln_f(x) # (B,T,C)
        logits = self.lm_head(x) # (B,T,vocab_size)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        # idx is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            # crop idx to the last block_size tokens
            idx_cond = idx[:, -block_size:]
            # get the predictions
            logits, loss = self(idx_cond)
            # focus only on the last time step
            logits = logits[:, -1, :] # becomes (B, C)
            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1) # (B, C)
            # sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            # append sampled index to the running sequence
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
        return idx

model = BigramLanguageModel()
m = model.to(device)
# print the number of parameters in the model
print(sum(p.numel() for p in m.parameters())/1e6, 'M parameters')

# create a PyTorch optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

for iter in range(max_iters):

    # every once in a while evaluate the loss on train and val sets
    if iter % eval_interval == 0 or iter == max_iters - 1:
        losses = estimate_loss()
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    # sample a batch of data
    xb, yb = get_batch('train')

    # evaluate the loss
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

# generate from the model
context = torch.zeros((1, 1), dtype=torch.long, device=device)
print(decode(m.generate(context, max_new_tokens=2000)[0].tolist()))
